# Namma Space — train a photoreal Gaussian splat of the hostel

This turns the hostel PoseCam capture into a **photoreal Gaussian splat** (like the home/kitchen scene).
Training needs an NVIDIA GPU, which free Colab provides.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.* Then *Runtime → Run all*.

What happens (≈ 45–75 min in total):
1. installs Nerfstudio 1.1.5 (the first training step also compiles the GPU rasteriser — several minutes, normal);
2. downloads the prepared dataset from the repository: 1,376 PoseCam frames with ARCore poses (no COLMAP needed) and a 300k-point starting cloud;
3. trains `splatfacto` for 30,000 steps in the hostel's own coordinate frame (metres, floor at y = 0);
4. measures quality on 137 held-out photos (PSNR);
5. exports `splat.ply` and copies it to your Google Drive → share that file's link with the team.

Nothing else needs to be set: the dataset was built with `python -m navkit nerfstudio` and checked against Nerfstudio's own loader.

In [ ]:
# 1. Check the GPU (should list a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install Nerfstudio (pins gsplat 1.4.0; its CUDA kernels compile on first use)
!pip install -q nerfstudio==1.1.5
!ns-train --help > /dev/null && echo "nerfstudio installed"

In [ ]:
# 3. Download the prepared hostel dataset (~45 MB)
!wget -q -O /content/hostel.zip https://github.com/atharv-namdeo/3d-mapping/raw/main/datasets/hostel_posecam_nerfstudio.zip
!unzip -q -o /content/hostel.zip -d /content/
!ls /content/hostel_posecam_nerfstudio && ls /content/hostel_posecam_nerfstudio/images | wc -l

### 4. Train (≈ 30–60 min on a T4)
The four `nerfstudio-data` flags keep PoseCam's metric, gravity-aligned frame so the result lines up with the
hostel's map, labels and routes with no manual alignment. Keep this tab open while it trains.

In [ ]:
!ns-train splatfacto \
    --data /content/hostel_posecam_nerfstudio \
    --output-dir /content/outputs \
    --experiment-name hostel \
    --max-num-iterations 30000 \
    --vis tensorboard \
    --viewer.quit-on-train-completion True \
    nerfstudio-data \
    --orientation-method none --center-method none --auto-scale-poses False --load-3D-points True

In [ ]:
# 5. Quality on the 137 held-out photos (higher PSNR = sharper; > 22 dB is good for 640x480 phone frames)
import glob, json, subprocess
config = sorted(glob.glob('/content/outputs/hostel/splatfacto/*/config.yml'))[-1]
print('config:', config)
subprocess.run(['ns-eval', '--load-config', config, '--output-path', '/content/eval.json'], check=True)
results = json.load(open('/content/eval.json'))['results']
print({k: round(v, 3) for k, v in results.items() if not k.endswith('_std')})

In [ ]:
# 6. Export the splat
import os, shutil
subprocess.run(['ns-export', 'gaussian-splat', '--load-config', config, '--output-dir', '/content/export'], check=True)
run_dir = os.path.dirname(config)
shutil.copy(os.path.join(run_dir, 'dataparser_transforms.json'), '/content/export/')
shutil.copy('/content/eval.json', '/content/export/')
!ls -la /content/export

In [ ]:
# 7. Copy to Google Drive -> open Drive, folder "namma_space_hostel_splat", share it ("Anyone with the link")
from google.colab import drive
drive.mount('/content/drive')
dest = '/content/drive/MyDrive/namma_space_hostel_splat'
os.makedirs(dest, exist_ok=True)
for f in ('splat.ply', 'dataparser_transforms.json', 'eval.json'):
    shutil.copy(os.path.join('/content/export', f), dest)
print('Saved to Google Drive:', dest)

### Next
Send the Drive link to whoever maintains the repo. The splat is imported with:
```
python -m navkit build app/scenes/hostel      # with "splatSource": "trained_splat.ply" in scene.json
```
The hostel then offers **Points** and a photoreal **Surfaces** view; labels, search and routes stay unchanged.

**Own NVIDIA GPU instead of Colab?** Run the same `ns-train …`, `ns-eval …` and `ns-export …` commands locally
(Nerfstudio 1.1.5 with CUDA).